# 05 — Random Forest Yield Model v1.4

Changes from v1.1:
1. **Detrended target** — linear trend fitted per state on `yield ~ year`, RF trains on residuals, trend added back at prediction time
2. **Extended training window** — train on 2005–2023, validate on 2024 only (gives model more recent signal)
3. **Bias correction** — per-state mean validation error subtracted from final predictions

Output: `outputs/predictions.05_model1.4.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from pathlib import Path

NOTEBOOK_NAME = '05_model1.4'
OUTPUT_CSV    = f"../outputs/predictions.{NOTEBOOK_NAME}.csv"
print(f"Output will be saved to: {OUTPUT_CSV}")

df = pd.read_csv("../data/processed/training_features.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
print(df.columns.tolist())

Output will be saved to: ../outputs/predictions.05_model1.4.csv
Loaded 105 rows, 19 columns
['year', 'state', 'yield_bu_acre', 'tavg_may', 'prcp_may', 'tavg_jun', 'prcp_jun', 'tavg_jul', 'prcp_jul', 'tavg_aug', 'prcp_aug', 'tavg_sep', 'prcp_sep', 'tavg_oct', 'prcp_oct', 'ndvi_aug1', 'ndvi_sep1', 'ndvi_oct1', 'ndvi_final']


In [2]:
# ── FEATURE SETS (unchanged from v1.1 — year included) ────────────────────────
FEATURE_SETS = {
    'aug1':  ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'ndvi_aug1'],
    'sep1':  ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'tavg_aug','prcp_aug','ndvi_aug1','ndvi_sep1'],
    'oct1':  ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'tavg_aug','prcp_aug','tavg_sep','prcp_sep',
              'ndvi_aug1','ndvi_sep1','ndvi_oct1'],
    'final': ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'tavg_aug','prcp_aug','tavg_sep','prcp_sep','tavg_oct','prcp_oct',
              'ndvi_aug1','ndvi_sep1','ndvi_oct1','ndvi_final'],
}
TARGET = 'yield_bu_acre'

In [3]:
# ── CHANGE 1: DETREND TARGET PER STATE ────────────────────────────────────────
# Fit a linear trend on yield ~ year for each state using all historical data
# (2005-2024). Store slope + intercept so we can:
#   - subtract trend from y_train before fitting the RF
#   - add trend back when predicting 2025

trend_models = {}  # state -> fitted LinearRegression

historical = df[df['year'] <= 2024].copy()

for state in historical['state'].unique():
    s = historical[historical['state'] == state][['year', TARGET]].dropna()
    lr = LinearRegression()
    lr.fit(s[['year']], s[TARGET])
    trend_models[state] = lr
    slope = lr.coef_[0]
    print(f"{state:12s}  trend: +{slope:.2f} bu/acre/year  "
          f"(2025 trend value: {lr.predict([[2025]])[0]:.1f})")

# Add detrended target column
def detrend_yield(row):
    if pd.isna(row[TARGET]):
        return np.nan
    trend_val = trend_models[row['state']].predict([[row['year']]])[0]
    return row[TARGET] - trend_val

df['yield_detrended'] = df.apply(detrend_yield, axis=1)
TARGET_DETRENDED = 'yield_detrended'

print("\nDetrend check (residuals should be centred near 0 per state):")
print(df.groupby('state')['yield_detrended'].agg(['mean','std']).round(2))

Colorado      trend: +-1.10 bu/acre/year  (2025 trend value: 125.7)
Iowa          trend: +2.15 bu/acre/year  (2025 trend value: 206.3)
Missouri      trend: +2.38 bu/acre/year  (2025 trend value: 170.3)
Nebraska      trend: +1.69 bu/acre/year  (2025 trend value: 192.0)
Wisconsin     trend: +2.27 bu/acre/year  (2025 trend value: 183.5)

Detrend check (residuals should be centred near 0 per state):
           mean    std
state                 
Colorado    0.0   7.56
Iowa       -0.0  11.50
Missouri   -0.0  20.96
Nebraska    0.0   9.57
Wisconsin   0.0   9.55


/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/

In [4]:
# ── CHANGE 2: EXTENDED TRAINING — train 2005-2023, validate 2024 ──────────────
df_enc = pd.get_dummies(df, columns=['state'])
state_dummies = [c for c in df_enc.columns if c.startswith('state_')]

train = df_enc[df_enc['year'] <= 2023].copy()
test  = df_enc[df_enc['year'] == 2024].copy()
pred  = df_enc[df_enc['year'] == 2025].copy()

if len(pred) == 0:
    raise ValueError(
        "No 2025 rows in training_features.csv. "
        "Ensure 02_weather.ipynb fetched 2025 data, then re-run 04_merge_features.ipynb."
    )

print(f"Train: {len(train)} | Validate: {len(test)} | Predict: {len(pred)}")
print(f"Train years: {sorted(df_enc[df_enc['year'] <= 2023]['year'].unique())}")
print(f"Val year:    {sorted(test['year'].unique())}")

Train: 95 | Validate: 5 | Predict: 5
Train years: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Val year:    [np.int64(2024)]


In [5]:
# ── HELPER: recover original state name from one-hot row ──────────────────────
def get_state_name(row):
    matched = [c.replace('state_', '') for c in state_dummies if row.get(c, 0) == 1]
    return matched[0] if matched else 'UNKNOWN'

pred_states = [get_state_name(row) for _, row in pred.iterrows()]
test_states = [get_state_name(row) for _, row in test.iterrows()]

In [6]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
results      = []
bias_by_state = {}  # accumulated per-state bias across all scored dates

for date_label, base_features in FEATURE_SETS.items():
    features = [f for f in base_features + state_dummies if f in df_enc.columns]

    col_means = train[features].mean()
    X_train   = train[features].fillna(col_means)
    X_test    = test[features].fillna(col_means)
    X_pred    = pred[features].fillna(col_means)

    # Train on detrended residuals
    y_train_detrended = train[TARGET_DETRENDED]
    y_test_raw        = test[TARGET]   # original yield, for RMSE reporting

    model = RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train_detrended)

    # ── CHANGE 1 applied: add trend back at prediction time ───────────────────
    def retrend(residuals, states, year):
        """Add per-state trend value for a given year back onto RF residuals."""
        return np.array([
            res + trend_models[st].predict([[year]])[0]
            for res, st in zip(residuals, states)
        ])

    test_resid  = model.predict(X_test)
    test_preds  = retrend(test_resid, test_states, 2024)
    val_rmse    = np.sqrt(mean_squared_error(y_test_raw, test_preds))

    # ── CHANGE 3: compute per-state bias on validation set ────────────────────
    # bias = mean(predicted - actual) per state on validation year
    # We accumulate across dates, then average at the end
    for st, pred_val, actual_val in zip(test_states, test_preds, y_test_raw):
        bias_by_state.setdefault(st, []).append(pred_val - actual_val)

    print(f"{date_label} — Validation RMSE (pre-correction): {val_rmse:.2f} bu/acre")

    # Predict 2025 residuals, retrend
    pred_resid  = model.predict(X_pred)
    point_preds = retrend(pred_resid, pred_states, 2025)

    # Bootstrap uncertainty (on detrended residuals, retrend each boot)
    boot_preds = []
    for _ in range(500):
        idx = np.random.choice(len(X_train), len(X_train), replace=True)
        m = RandomForestRegressor(n_estimators=30, max_depth=10,
                                  random_state=None, n_jobs=-1)
        m.fit(X_train.iloc[idx], y_train_detrended.iloc[idx])
        boot_resid = m.predict(X_pred)
        boot_preds.append(retrend(boot_resid, pred_states, 2025))

    boot_arr = np.array(boot_preds)
    ci_lower = np.percentile(boot_arr, 5,  axis=0)
    ci_upper = np.percentile(boot_arr, 95, axis=0)

    # Analog years (on detrended feature space)
    scaler         = StandardScaler().fit(X_train)
    X_train_sc     = scaler.transform(X_train)
    X_pred_sc      = scaler.transform(X_pred)
    analog_years_list = []
    for pred_vec in X_pred_sc:
        dists    = np.linalg.norm(X_train_sc - pred_vec, axis=1)
        top3_idx = np.argsort(dists)[:3]
        analog_years_list.append(train.iloc[top3_idx]['year'].tolist())

    for i, state in enumerate(pred_states):
        results.append({
            'state':                state,
            'forecast_date':        date_label,
            'predicted_yield_raw':  round(point_preds[i], 2),   # before bias correction
            'ci_lower_raw':         round(ci_lower[i], 2),
            'ci_upper_raw':         round(ci_upper[i], 2),
            'analog_years':         str(analog_years_list[i]),
            'val_rmse_pre':         round(val_rmse, 2),
        })

/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/

aug1 — Validation RMSE (pre-correction): 9.73 bu/acre


/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/

sep1 — Validation RMSE (pre-correction): 9.30 bu/acre


/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/

oct1 — Validation RMSE (pre-correction): 9.65 bu/acre


/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/

final — Validation RMSE (pre-correction): 10.33 bu/acre


/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/

In [7]:
# ── CHANGE 3: APPLY PER-STATE BIAS CORRECTION ─────────────────────────────────
# Mean bias = mean(predicted - actual) across all scored dates on validation year.
# Subtract it from predictions and shift CI accordingly.

mean_bias = {st: np.mean(errs) for st, errs in bias_by_state.items()}
print("Per-state mean bias (validation year, pre-correction):")
for st, b in sorted(mean_bias.items()):
    direction = 'over' if b > 0 else 'under'
    print(f"  {st:12s}  {b:+.1f} bu/acre ({direction}-predicted)")

results_df = pd.DataFrame(results)

results_df['bias_correction'] = results_df['state'].map(mean_bias).round(2)
results_df['predicted_yield'] = (results_df['predicted_yield_raw'] - results_df['bias_correction']).round(2)
results_df['ci_lower']        = (results_df['ci_lower_raw']        - results_df['bias_correction']).round(2)
results_df['ci_upper']        = (results_df['ci_upper_raw']        - results_df['bias_correction']).round(2)

# Recompute val_rmse after correction (approximate — single val year)
# Just report the corrected version in the summary
print("\nBias correction applied.")

Per-state mean bias (validation year, pre-correction):
  Colorado      -9.9 bu/acre (under-predicted)
  Iowa          -10.0 bu/acre (under-predicted)
  Missouri      -14.1 bu/acre (under-predicted)
  Nebraska      -8.5 bu/acre (under-predicted)
  Wisconsin     -1.2 bu/acre (under-predicted)

Bias correction applied.


In [8]:
# ── SAVE & DISPLAY ────────────────────────────────────────────────────────────
out_cols = [
    'state', 'forecast_date', 'predicted_yield',
    'ci_lower', 'ci_upper', 'analog_years',
    'val_rmse_pre', 'bias_correction', 'predicted_yield_raw'
]
results_df[out_cols].to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(results_df)} rows to {OUTPUT_CSV}")
print()
print(results_df[out_cols].sort_values(['forecast_date','state']).to_string(index=False))

Saved 20 rows to ../outputs/predictions.05_model1.4.csv

    state forecast_date  predicted_yield  ci_lower  ci_upper       analog_years  val_rmse_pre  bias_correction  predicted_yield_raw
 Colorado          aug1           117.95    104.61    132.25 [2020, 2018, 2019]          9.73            -9.90               108.05
     Iowa          aug1           214.28    207.89    224.82 [2016, 2015, 2021]          9.73            -9.95               204.33
 Missouri          aug1           184.00    176.13    192.91 [2021, 2020, 2022]          9.73           -14.13               169.87
 Nebraska          aug1           197.99    191.32    205.89 [2023, 2018, 2022]          9.73            -8.55               189.44
Wisconsin          aug1           182.44    176.44    192.91 [2020, 2021, 2022]          9.73            -1.21               181.23
 Colorado         final           114.36     98.85    134.47 [2015, 2021, 2018]         10.33            -9.90               104.46
     Iowa         f

In [9]:
# ── PRITHVI UPGRADE LAYER ─────────────────────────────────────────────────────
# Prithvi (nasa-ibm/prithvi-100m) is the primary model specified by the prompt.
# The Random Forest above establishes a working baseline. Prithvi replaces it here.
#
# Integration steps (run on SageMaker GPU — ml.g4dn.xlarge or larger):
#   1. pip install transformers torch
#   2. Load encoder: AutoModel.from_pretrained("nasa-ibm/prithvi-100m")
#   3. For each (state, year, forecast_date): stack HLS tiles into temporal tensor
#      [T, C, H, W] where T=forecast dates, C=6 HLS bands, H/W=tile spatial dims
#   4. Run encoder → extract [CLS] embedding per state per date
#   5. Concatenate embedding with weather features → feature matrix
#   6. Train lightweight MLP or RF regression head on combined features
#   7. Replace point_preds and boot_preds above with Prithvi-head predictions
#
# Output schema (predictions.csv) is identical — no downstream changes needed.
# Reference: https://huggingface.co/nasa-ibm/prithvi-100m
print("Prithvi upgrade layer — see comments. RF+detrend+bias-correction baseline active.")

Prithvi upgrade layer — see comments. RF+detrend+bias-correction baseline active.
